In [26]:
import csv
import importlib
import os
import random
import sys
import torch
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import pandas as pd

from time import sleep
from collections import deque
from itertools import count
from typing import Any, Dict, List, Optional, Tuple, Set
from collections import defaultdict

sources_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if sources_path not in sys.path:
    sys.path.append(sources_path)

from importnb import Notebook
with Notebook():
    from Labs.LatencyModel import LatencyModel, MultiDULatencyModel
    from Labs.CacheEngine import CacheEngineEnv
    from Labs.UserRequest import UserRequestEvents
    from Labs.EnvWrapperLru import EnvWrapper

from RL.Adapters import FeatureAdapter, NetworkAdapter

import Common.config as config
import Common.datatypes as datatypes
import Common.debugger as debugger
import Common.utils as utils

importlib.reload(config)
importlib.reload(datatypes)
importlib.reload(debugger)
importlib.reload(utils)

<module 'Common.utils' from 'c:\\Users\\es25591\\Workspace\\CacheVideoPredict360\\Sources\\Common\\utils.py'>

In [27]:
CachePolicy = datatypes.CachePolicy
CacheKey = datatypes.CacheKey  # (vid, layer, tile, gop)

cfg = config.Config()

cfg.filename = f"lru.csv"
cfg.n_episodes = 100

debugger = debugger.debug

top_k = int(getattr(cfg, "topk_content_plot_k", 20))

In [28]:
class LruPolicy(CachePolicy):

    def __init__(self, max_videos: int = 50, cfg: Any = None) -> None:
        self.cfg = cfg
        self.cur_size = 0

        # Video LRU: oldest video (by any access to its subrequests) at index 0
        self.video_access_order: List[int] = []

        # Tile LRU per video: video -> tiles ordered from oldest to newest
        self.tile_access_order: Dict[int, List[int]] = {}

        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]

    def _touch_video(self, video: int) -> None:
        if video in self.video_access_order:
            self.video_access_order.remove(video)
        self.video_access_order.append(video)

    def _touch_tile(self, video: int, tile: int) -> None:
        tiles = self.tile_access_order.setdefault(video, [])
        if tile in tiles:
            tiles.remove(tile)
        tiles.append(tile)

    def _ensure_video_slot(self, video: int) -> int:
        if video in self.video_idx:
            return self.video_idx.index(video)

        slot = self.video_idx.index(-1) if -1 in self.video_idx else None
        if slot is None:
            if not self.video_access_order:
                raise RuntimeError("LRU state is inconsistent: no free slot and no video to evict.")
            lru_video = self.video_access_order.pop(0)
            slot = self.video_idx.index(lru_video)
            self.tile_access_order.pop(lru_video, None)

        self.video_idx[slot] = video
        self.tile_idx[slot] = [-1] * self.cfg.viewport
        return slot

    def put(
        self, 
        key: int, 
        value: Any, 
        size: int
    ):
        new_video, new_tile = value
        slot = self._ensure_video_slot(new_video)

        self._touch_video(new_video)

        if new_tile == -1:
            self.tile_access_order.pop(new_video, None)
            self.tile_idx[slot] = [-1] * self.cfg.viewport
        else:
            self._touch_tile(new_video, new_tile)
            slot_tiles = self.tile_idx[slot]
            if new_tile in slot_tiles:
                slot_tiles.remove(new_tile)
            slot_tiles.append(new_tile)
            if len(slot_tiles) > self.cfg.viewport:
                slot_tiles.pop(0)

        self.cur_size = sum(1 for v in self.video_idx if v != -1)
        return []

    def get(self, key: CacheKey) -> Optional[Any]:
        raise NotImplementedError("LRU policy does not support manual retrieval of individual keys.")

    def contains(self, key: CacheKey) -> bool:
        raise NotImplementedError("LRU policy does not support manual checking of individual keys.")
    
    def keys(self):
        raise NotImplementedError("LRU policy does not support manual retrieval of keys.")
    
    def remove(self, key: CacheKey) -> bool:
        raise NotImplementedError("LRU policy does not support manual removal of individual keys.")
    
    def get_capacity(self) -> int:
        return self.cur_size
    
    def clear(self) -> None:
        self.video_access_order.clear()
        self.tile_access_order.clear()
        self.cur_size = 0
        self.video_idx = [-1] * self.cfg.cache_size
        self.tile_idx = [[-1] * self.cfg.viewport for _ in range(self.cfg.cache_size)]

    def stats(self) -> Dict[str, Any]:
        return {
            'current_size': self.cur_size,
            'capacity': self.cfg.cache_size,
            'num_items': len([v for v in self.video_idx if v != -1])
        }

In [29]:
def save_training_results(
    path_,
    filename,
    ep, 
    total_reward, 
    cache_hits, 
    cache_misses, 
    agent
):
    with open(os.path.join(path_, filename), 'a', newline='') as f:
        fieldnames = [
            'episode', 
            'total_reward', 
            'cache_hits', 
            'cache_misses', 
            'epsilon',
            'lr'
        ]
        writer_results = csv.DictWriter(f, fieldnames=fieldnames)

        if ep == 0:
            writer_results.writeheader()
        
        writer_results.writerow({
            'episode': ep,
            'total_reward': round(float(total_reward), 2),
            'cache_hits': cache_hits,
            'cache_misses': cache_misses,
            'lr': f"{agent.scheduler.get_last_lr()[0]:.10f}" if agent else None,
            'epsilon': round(float(agent.epsilon), 4) if agent else None
        })

def _append_csv_row(csv_path: str, fieldnames: list[str], row: dict) -> None:
    write_header = not os.path.exists(csv_path)
    with open(csv_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerow(row)

def save_episode_metrics(
    metrics_dir: str,
    ep: int,
    total_reward: float,
    cache_hits: int,
    cache_misses: int,
    chr: float,
    psnr: float,
    agent,
):
    csv_path = os.path.join(metrics_dir, 'episode_metrics.csv')
    fieldnames = [
        'episode',
        'total_reward',
        'cache_hits',
        'cache_misses',
        'hit_rate',
        'chr',
        'psnr',
        'epsilon',
        'lr',
    ]

    row = {
        'episode': ep,
        'total_reward': round(float(total_reward), 2),
        'cache_hits': cache_hits,
        'cache_misses': cache_misses,
        'hit_rate': float(cache_hits) / float(cache_hits + cache_misses + 1e-9),
        'chr': round(float(chr), 6) if chr is not None else None,
        'psnr': round(float(psnr), 6) if psnr is not None else None,
        'epsilon': round(float(agent.epsilon), 6) if agent else None,
        'lr': float(agent.scheduler.get_last_lr()[0]) if agent else None,
    }
    _append_csv_row(csv_path, fieldnames, row)

def update_metrics(info: dict, reward: float) -> tuple[float, int, int, int, int]:
    enh_hits = info.get("enh_layer_hits", 0)
    base_hits = info.get("base_layer_hits", 0)
    enh_misses = info.get("enh_layer_misses", 0)
    base_misses = info.get("base_layer_misses", 0)

    return reward, base_hits, base_misses, enh_hits, enh_misses

def _print_video_psnr_ranking(
    video_watch_counts: dict[int, int],
    video_psnr_sums: dict[int, float],
    video_psnr_counts: dict[int, int],
    top_k: int,
) -> None:
    if not video_watch_counts:
        print("No watched videos recorded for this episode.")
        return

    ranked_videos = sorted(
        video_watch_counts.items(),
        key=lambda item: item[1],
        reverse=True,
    )[:max(1, int(top_k))]

    print(f"Mean PSNR across watched videos: {_mean_video_psnr(video_psnr_sums, video_psnr_counts):.2f}")
    print(f"Top {len(ranked_videos)} watched videos by rank:")
    for rank, (video_id, watch_count) in enumerate(ranked_videos, start=1):
        mean_psnr = 0.0
        if video_psnr_counts.get(video_id, 0) > 0:
            mean_psnr = video_psnr_sums[video_id] / video_psnr_counts[video_id]
        print(
            f"{rank:02d}. Video {video_id} | watches={watch_count} | mean_psnr={mean_psnr:.2f}"
        )

def _mean_video_psnr(video_psnr_sums: dict[int, float], video_psnr_counts: dict[int, int]) -> float:
    per_video_psnr = [
        video_psnr_sums[video_id] / video_psnr_counts[video_id]
        for video_id in video_psnr_counts
        if video_psnr_counts[video_id] > 0
    ]
    return float(np.mean(per_video_psnr)) if per_video_psnr else 0.0

def _snapshot_cache(env) -> set[tuple[int, int]]:
    cache_set: set[tuple[int, int]] = set()
    cache_entries = getattr(env.mec_cache.policy, "cache", [])

    for video, tile in cache_entries:
        if int(video) == -1:
            continue
        cache_set.add((int(video), int(tile)))

    return cache_set

def _content_key(video: int, tile: int | None) -> tuple[int, int]:
    return (int(video), -1 if tile is None else int(tile))

def build_latency_model(cfg):
    """Build and return the MultiDULatencyModel."""
    P = cfg.n_nodes
    max_U = cfg.n_users

    return MultiDULatencyModel(
        P=P,
        max_U=max_U,
        R_M_D=80e6,
        R_C_M=125e6,
        mu=2e7,
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 20e6, dtype=float),
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float),
        rhoT_p=[0.2],
        lambda_p=[0.05],
        du_fixed_delay=0.001,
        mec_fixed_delay=0.005,
        cloud_fixed_delay=0.1
    )

def build_environment(cfg):
    """Construct the full multi-component environment wrapper."""
    du_caches = []

    policy=LruPolicy(
        cfg=cfg,
        max_videos=cfg.cache_size,
    )

    # MEC Cache Engine
    mec_cache = CacheEngineEnv(
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n_gops=cfg.n_gops,
        cache_capacity=cfg.cache_capacity,
        policy=policy
    )

    # User request generator
    users_env = UserRequestEvents(
        n_nodes=cfg.n_nodes,
        n_users=cfg.n_users,
        n_videos=cfg.n_videos,
        n_gops=cfg.n_gops,
        n_layers=cfg.n_layers,
        n_tiles=cfg.n_tiles,
        n=cfg.n,
        m=cfg.m,
        arrival_rate=cfg.arrival_rate,
        zipf_alpha=cfg.zipf_alpha
    )

    # Latency Model
    latency_model = build_latency_model(cfg)
    
    # Wrapping all into the main training environment
    return EnvWrapper(
        cfg=cfg,
        n=cfg.n,
        m=cfg.m,
        n_layers=cfg.n_layers,
        users_env=users_env,
        du_caches=du_caches,
        mec_cache=mec_cache,
        latency_model=latency_model,
        theta=cfg.theta,
        lam=cfg.lam,
        max_steps=cfg.max_steps,
        prefetch_fn=lambda cache, action: cache.lru_live_prefetching(action),
        reward_fn=lambda env, reqs: env.compute_reward(reqs),
        debugger=debugger
    )

In [30]:

def run_episode(episode, env, net_adapter, cfg, metrics_dir, global_step_start):
    """Run one full training episode."""
    _, info = net_adapter.reset()

    total_reward = 0.0
    cache_hits = cache_misses = 0
    base_hits = base_misses = 0
    enh_hits = enh_misses = 0
    psnr_sum = 0.0
    popularity_bonus_sum = 0.0

    episode_request_counts: dict[tuple[int, int], int] = defaultdict(int)
    video_watch_counts: dict[int, int] = defaultdict(int)
    video_psnr_sums: dict[int, float] = defaultdict(float)
    video_psnr_counts: dict[int, int] = defaultdict(int)

    global_step = global_step_start

    if cfg.has_warmup and episode == 0:
        env.warmup_phase(net_adapter, 1000)

    for step in range(cfg.max_steps):
        global_step += 1
        req_state = info.get("user_request", None)
        
        current_video_id = None

        if req_state is not None:
            current_video_id = int(req_state["video"])
            episode_request_counts[_content_key(current_video_id, None)] += 1
            video_watch_counts[current_video_id] += 1
            for tile in req_state.get("viewport", []):
                episode_request_counts[_content_key(current_video_id, int(tile))] += 1

        _, reward, _, info = env.step(req_state, net_adapter)

        if current_video_id is not None:
            step_psnr = float(info.get("psnr", 0.0))
            video_psnr_sums[current_video_id] += step_psnr
            video_psnr_counts[current_video_id] += 1

        delta_r, bs_hits, bs_miss, e_hits, e_miss = update_metrics(info, reward)
        total_reward += delta_r
        cache_hits += bs_hits + e_hits
        cache_misses += bs_miss + e_miss
        base_hits += bs_hits
        base_misses += bs_miss
        enh_hits += e_hits
        enh_misses += e_miss
        psnr_sum += info.get("psnr", 0.0)

        if net_adapter.env_is_done():
            break

        debugger.log("cache_hits", cache_hits)
        debugger.log("cache_misses", cache_misses)

    cache_snapshot = _snapshot_cache(env)

    return (
        total_reward,
        cache_hits,
        cache_misses,
        base_hits,
        base_misses,
        enh_hits,
        enh_misses,
        global_step,
        psnr_sum / (step + 1),
        dict(episode_request_counts),
        dict(video_watch_counts),
        dict(video_psnr_sums),
        dict(video_psnr_counts),
        cache_snapshot,
        popularity_bonus_sum,
    )

def train(cfg):
    print("\n--- Starting DRL Caching System ---")

    env = build_environment(cfg)

    feature_adapter = FeatureAdapter(cfg, env)
    net_adapter = NetworkAdapter(cfg, env, feature_adapter)

    date_dir = pd.Timestamp.now().strftime("%Y-%m-%d_%H-%M")
    debug_path = os.path.join(cfg.path_results, date_dir)
    metrics_dir = os.path.join(debug_path, "metrics")
    os.makedirs(metrics_dir, exist_ok=True)

    global_log_path = os.path.join(cfg.path_results, "global.log")
    with open(global_log_path, "a", encoding="utf-8") as f:
        f.write(f"{pd.Timestamp.now().isoformat()} | Max Steps: {cfg.max_steps} | ")
        f.write("-" * 50 + "\n")

    global_step = 0

    for episode in range(cfg.n_episodes):
        (
            total_reward,
            cache_hits,
            cache_misses,
            base_hits,
            base_misses,
            enh_hits,
            enh_misses,
            global_step,
            psnr_rate,
            episode_request_counts,
            episode_video_watch_counts,
            episode_video_psnr_sums,
            episode_video_psnr_counts,
            cache_snapshot,
            popularity_bonus_sum,
        ) = run_episode(episode, env, net_adapter, cfg, metrics_dir, global_step)

        debugger.save_results(filepath=f"{debug_path}/json/debug_ep{episode}")
        debugger.clear()

        base_hit_rate = base_hits / (base_hits + base_misses + 1e-9)
        enh_hit_rate = enh_hits / (enh_hits + enh_misses + 1e-9)
        mean_video_psnr = _mean_video_psnr(episode_video_psnr_sums, episode_video_psnr_counts)

        print(
            f"Episode {episode} | R: {int(total_reward)} | "
            f"HR: {cache_hits / (cache_hits + cache_misses + 1e-9):.2f} | "
            f"HR_2: {(base_hit_rate + enh_hit_rate * 4) / 5:.2f} | "
            f"BHR: {base_hit_rate:.2f} | "
            f"EHR: {enh_hit_rate:.2f} | "
            f"PSNR: {psnr_rate:.2f} | "
            f"Mean video PSNR: {mean_video_psnr:.2f} | "
            f"Popularity bonus: {popularity_bonus_sum:.4f} | "
            f"Time: {pd.Timestamp.now().strftime('%H:%M:%S')}"
        )

        _print_video_psnr_ranking(
            video_watch_counts=episode_video_watch_counts,
            video_psnr_sums=episode_video_psnr_sums,
            video_psnr_counts=episode_video_psnr_counts,
            top_k=top_k,
        )
        print("-" * 50)

        save_episode_metrics(
            metrics_dir=metrics_dir,
            ep=episode,
            total_reward=total_reward,
            cache_hits=cache_hits,
            cache_misses=cache_misses,
            chr=(base_hit_rate + enh_hit_rate * 4) / 5,
            psnr=mean_video_psnr,
            agent=None
        )

if __name__ == "__main__":
    train(cfg)


--- Starting DRL Caching System ---
NetworkAdapter initialized with capacity: 100 videos, 4 tiles per video
Episode 0 | R: 0 | HR: 0.83 | HR_2: 0.59 | BHR: 0.93 | EHR: 0.51 | PSNR: 35.80 | Mean video PSNR: 35.79 | Popularity bonus: 0.0000 | Time: 12:36:49
Mean PSNR across watched videos: 35.79
Top 20 watched videos by rank:
01. Video 0 | watches=1240 | mean_psnr=35.60
02. Video 1 | watches=842 | mean_psnr=35.81
03. Video 4 | watches=580 | mean_psnr=35.84
04. Video 2 | watches=378 | mean_psnr=36.12
05. Video 3 | watches=372 | mean_psnr=35.76
06. Video 7 | watches=318 | mean_psnr=35.92
07. Video 5 | watches=281 | mean_psnr=35.58
08. Video 11 | watches=213 | mean_psnr=36.36
09. Video 9 | watches=206 | mean_psnr=36.44
10. Video 6 | watches=203 | mean_psnr=35.33
11. Video 12 | watches=168 | mean_psnr=36.34
12. Video 8 | watches=165 | mean_psnr=36.20
13. Video 14 | watches=154 | mean_psnr=35.55
14. Video 16 | watches=130 | mean_psnr=34.73
15. Video 13 | watches=125 | mean_psnr=36.08
16. Vid